In [ ]:
# Setup environment and clone repository
import os
import sys

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    print("Running on Kaggle")
    
    # Clone the repository if not already present
    if not os.path.exists('/kaggle/working/NLP_PROJECT_2025'):
        print("Cloning repository from mohab branch...")
        !git clone -b mohab https://github.com/MohabYasser2/NLP_PROJECT_2025.git /kaggle/working/NLP_PROJECT_2025
        print("✓ Repository cloned!")
    else:
        print("✓ Repository already exists, pulling latest changes...")
        !cd /kaggle/working/NLP_PROJECT_2025 && git fetch origin mohab && git reset --hard origin/mohab
        print("✓ Repository updated to latest version!")
    
    # Add to Python path
    sys.path.insert(0, '/kaggle/working/NLP_PROJECT_2025')
else:
    print("Running locally")
    # Add parent directory to path for local execution
    sys.path.insert(0, os.path.abspath('..'))

print(f"✓ Python path configured: {sys.path[0]}")

In [ ]:
# Detect available datasets on Kaggle
if IS_KAGGLE:
    print("\n=== Available Kaggle Input Datasets ===")
    !ls -la /kaggle/input/
    print("\n=== Looking for data files ===")
    import glob
    data_files = glob.glob('/kaggle/input/**/*.txt', recursive=True)
    print(f"Found .txt files: {data_files}")
    
    # Try to auto-detect the dataset directory
    if data_files:
        dataset_dir = os.path.dirname(data_files[0])
        print(f"\n✓ Auto-detected dataset directory: {dataset_dir}")
        
        # List all files in dataset directory
        all_files = [os.path.basename(f) for f in data_files if os.path.dirname(f) == dataset_dir]
        print(f"Available files: {all_files}")
        
        # Auto-detect file names
        TRAIN_FILE = os.path.join(dataset_dir, 'train.txt')
        
        # Check for dev/val file
        if 'val.txt' in all_files:
            DEV_FILE = os.path.join(dataset_dir, 'val.txt')
            print("Using 'val.txt' for validation")
        elif 'dev.txt' in all_files:
            DEV_FILE = os.path.join(dataset_dir, 'dev.txt')
            print("Using 'dev.txt' for validation")
        else:
            print("⚠ Warning: No val.txt or dev.txt found, using train.txt for validation")
            DEV_FILE = TRAIN_FILE
        
        TEST_FILE = os.path.join(dataset_dir, 'test.txt')
    else:
        print("\n⚠ No .txt files found! Please add your dataset to this notebook.")
        print("Click 'Add Data' → Search for your dataset → Add to notebook")
else:
    TRAIN_FILE = '../data/train.txt'
    DEV_FILE = '../data/val.txt'
    TEST_FILE = '../data/test.txt'

OUTPUT_FILE = '/kaggle/working/submission.csv' if IS_KAGGLE else 'submission.csv'

print(f"\n=== Final Configuration ===")
print(f"Train file: {TRAIN_FILE} (exists: {os.path.exists(TRAIN_FILE) if 'TRAIN_FILE' in locals() else 'N/A'})")
print(f"Dev file: {DEV_FILE} (exists: {os.path.exists(DEV_FILE) if 'DEV_FILE' in locals() else 'N/A'})")
print(f"Test file: {TEST_FILE} (exists: {os.path.exists(TEST_FILE) if 'TEST_FILE' in locals() else 'N/A'})")
print(f"Output file: {OUTPUT_FILE}")

## Import Training Module

All the heavy implementation is in the `/src` directory. This notebook just calls the training function.

In [ ]:
# Import training function
from src.training.train_crf import run_crf_training

print("✓ Imports successful")

## Train CRF Model

This will:
1. Load and preprocess data
2. Extract rich contextual features for CRF
3. Train CRF using forward-backward algorithm
4. Evaluate on dev set with Viterbi decoding
5. Generate predictions for test set
6. Save predictions to CSV

**Note**: CRF training may take 15-30 minutes due to forward-backward computations.

In [ ]:
print("⚡ ULTRA-LIGHT MODE: Extreme memory optimization")
print("=" * 60)
print("⚠ Using minimal dataset (10k samples) to prevent crash")

# CRITICAL: Subsample training data to fit in memory
import random
random.seed(42)

# Load and subsample training data
from pathlib import Path
with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    all_train = f.readlines()

# Use only 20% of training data (10k instead of 50k)
sample_size = min(10000, len(all_train))
sampled_train = random.sample(all_train, sample_size)

# Create temporary subsampled file
TRAIN_FILE_SMALL = '/kaggle/working/train_small.txt'
with open(TRAIN_FILE_SMALL, 'w', encoding='utf-8') as f:
    f.writelines(sampled_train)

print(f"✓ Subsampled training data: {len(all_train)} → {sample_size} sentences")

# Run with extreme memory optimizations
run_crf_training(
    train_file=TRAIN_FILE_SMALL,  # Use subsampled data
    dev_file=DEV_FILE,
    test_file=TEST_FILE,
    output_file=OUTPUT_FILE,
    max_iter=10,           # ↓ Even fewer iterations
    learning_rate=0.15,    # ↑ Even faster convergence
    regularization=2.0     # ↑ Very strong regularization
)

print("\n✓ Ultra-light training completed!")
print(f"✓ Submission saved to: {OUTPUT_FILE}")

## Verify Submission File

Let's check the first few lines of the submission file.

In [ ]:
import csv

print("Submission file preview:")
print("-" * 80)

with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        if i < 10:  # Show first 10 lines
            print(f"{row[0]}: {row[1][:50]}..." if len(row[1]) > 50 else f"{row[0]}: {row[1]}")
        else:
            break

print("-" * 80)
print("✓ Submission file is ready for upload!")

## Model Details

### CRF Architecture
- **Type**: Linear-chain CRF for sequence labeling
- **Features**: Contextual character features
  - Current character
  - Previous 1-2 characters  
  - Next 1-2 characters
  - Character bigrams and trigrams
  - Character type (letter/space/digit)
  - Word boundaries (start/end)
- **Transitions**: Learnable transition scores between all label pairs

### Training Algorithm
1. **Forward Algorithm**: Compute forward probabilities (α) in log-space
2. **Backward Algorithm**: Compute backward probabilities (β) in log-space
3. **Gradient Computation**: 
   - Expected feature counts from model (using α, β)
   - Observed feature counts from data
   - Gradient = observed - expected
4. **Parameter Update**: Gradient ascent with L2 regularization

### Inference
- **Viterbi Decoding**: Find most likely label sequence
- Dynamic programming to find optimal path

### Parameters
- Learning rate: 0.01
- Max iterations: 50
- L2 penalty: 0.1

### Implementation
All code is written from scratch using only NumPy:
- Forward-backward algorithm in log-space
- Gradient computation for CRF
- Viterbi decoding
- Feature extraction from character sequences

**No external ML libraries used!**